# Visor del inventario documental

Notebook de solo lectura para revisar `outputs/01_inventario/inventario_documental.csv`, incluidos los metadatos y vínculos con actuaciones aprobadas.

In [ ]:
from pathlib import Path

import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 120)

ROOT = Path.cwd()
if not (ROOT / 'outputs').exists():
    ROOT = ROOT.parent

INVENTARIO = ROOT / 'outputs' / '01_inventario' / 'inventario_documental.csv'
assert INVENTARIO.is_file(), f'No se encuentra el inventario: {INVENTARIO}'
inventario = pd.read_csv(INVENTARIO, dtype='string', keep_default_na=False)
inventario.head()

In [ ]:
resumen = pd.DataFrame({
    'documentos': [len(inventario)],
    'areas_logicas': [inventario['area'].nunique()],
    'tipos_documentales': [inventario['tipo_documental'].nunique()],
    'vinculos_automaticos': [(inventario['revision_accion'] == 'automatico').sum()],
    'vinculos_pendientes': [(inventario['revision_accion'] == 'pendiente_revision').sum()],
    'sin_vincular': [(inventario['revision_accion'] == 'sin_vincular').sum()],
})
resumen

In [ ]:
por_area = (inventario.groupby('area', dropna=False)
            .agg(documentos=('ruta_relativa', 'count'),
                 vinculados=('accion_aprobada_id', lambda serie: serie.ne('').sum()),
                 texto_extraido=('texto_preview', lambda serie: serie.ne('').sum()))
            .sort_values('documentos', ascending=False))
por_area

## Documentación de gastos

La siguiente vista permite revisar las facturas, nóminas y documentos relacionados dentro de `CONTABILIDAD/GASTOS` antes de completar la relación contable.

In [ ]:
gastos = inventario[inventario['ruta_relativa'].str.contains('/CONTABILIDAD/GASTOS/', regex=False)].copy()
columnas_gastos = [
    'ruta_relativa', 'nombre', 'tipo_documental', 'fechas_detectadas',
    'importes_detectados', 'accion_aprobada_id', 'confianza_accion',
    'revision_accion', 'texto_preview',
]
gastos[columnas_gastos].sort_values(['revision_accion', 'nombre'])

In [ ]:
gastos.groupby(['tipo_documental', 'revision_accion'], dropna=False).size().rename('documentos').to_frame()

In [ ]:
busqueda = 'mercartes'  # Sustituir por proveedor, numero de factura, concepto o actuacion.
coincidencias = inventario[
    inventario[['nombre', 'ruta_relativa', 'texto_preview']]
    .fillna('')
    .apply(lambda fila: fila.str.contains(busqueda, case=False, regex=False).any(), axis=1)
]
coincidencias[['ruta_relativa', 'nombre', 'tipo_documental', 'accion_aprobada_id', 'revision_accion', 'texto_preview']]